In [0]:
# Parâmetros via Databricks Widgets (1 notebook, 3 jobs com params diferentes)
# Para Teste
# precisa configurar IPCA, SELIC e CDI no WORKFLOW
dbutils.widgets.text("serie_codigo", "12",           "Código da Série BCB")
dbutils.widgets.text("serie_nome",   "cdi",   "Nome da Série")
dbutils.widgets.text("serie_freq",   "diario",       "Frequência: diaria|mensal|anual")

In [0]:
import time
import logging
from datetime import datetime
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType
from config import ROUTES, PipelineConfig

## Dados BCB - Indicadores de Desempenho 

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# config
SERIE_CODIGO = dbutils.widgets.get("serie_codigo")
SERIE_NOME   = dbutils.widgets.get("serie_nome")
SERIE_FREQ   = dbutils.widgets.get("serie_freq")
NOME_TABELA  = f"bronze_{SERIE_NOME}_{SERIE_FREQ}"  
BRONZE_PATH  = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

In [0]:
SCHEMA_BCB = StructType([
    StructField("data", StringType(), True),
    StructField("valor", StringType(), True),
])


# 1. Extração
hoje    = datetime.today()
inicio  = hoje.replace(year=hoje.year - 10).strftime("%d/%m/%Y")
url     = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{SERIE_CODIGO}/dados?formato=json&dataInicial={inicio}"

log.info(f"Iniciando ingestão | série={SERIE_CODIGO} | nome={SERIE_NOME} | frequencia={SERIE_FREQ} data_processamento={DATA_PROC}")

data = PipelineConfig.retorno_json_url(url=url)

log.info(f"Registros recebidos da API: {len(data)}")

# 2. Criação do DF com esquema definido
df = spark.createDataFrame(data, schema=SCHEMA_BCB)

# 3. Metadados de rastreabilidade 
df = (df
      .withColumn("_source_url", f.lit(url))
      .withColumn("_ingest_timestamp", f.current_timestamp())
      .withColumn("data_processamento", f.lit(DATA_PROC))
)


In [0]:
# 4. Escrita na bronze
n_registros = df.count()

log.info(f"Escrevendo {n_registros} linhas em Bronze ")

(df.coalesce(1)
    .write
    .mode("overwrite")
    .option("replaceWhere", f"data_processamento = {DATA_PROC}")
    .option("mergeSchema",  "true")
    .partitionBy("data_processamento")
    .format("delta")
    .saveAsTable(BRONZE_PATH)
 )


# 5. metricas
log.info(f"Ingestão concluida | série={SERIE_NOME} | caminho={BRONZE_PATH}  | linhas={n_registros} | partição={DATA_PROC}")

In [0]:
%sql 
select
    data_processamento
    ,count(*)
from 
    workspace.case_spark_cvm.bronze_cdi_diario_diaria
group by 1
